In [2]:
import json
import pandas as pd

pd.options.display.width = 300
pd.options.display.max_rows = 1500
pd.options.display.max_columns = None
pd.options.display.max_colwidth = None
pd.options.display.expand_frame_repr = False

fn = "../res/res.jsonl"
df = pd.read_json(fn, orient="records", lines=True)

# df = pd.read_json(path_or_buf=f, lines=True)

df['strategy'] = df['scriptName'].str.replace(' ', '\xa0')

# Только полезные поля
df = df[[
    "strategy", "imageUrl", "timeframe", "instrument", "sharpeRatio", "sortinoRatio",
    "maxStrategyDrawDownPercent", "buyHoldReturnPercent", "openPLPercent",
    "percentProfitable", "netProfitPercent", "profitFactor", "ratioAvgWinAvgLoss",
    "totalOpenTrades", "totalTrades", "totalTradesLong", "netProfitPercentLong",
    "profitFactorLong", "totalTradesShort", "netProfitPercentShort", "profitFactorShort",
    "avgSimTrade", "inTradeTimePercent", "r2", "firstTradeSize", "altMaxDrawDownPercent",
]]

# Формат значений
df = df.apply(pd.to_numeric, errors='ignore')
# df["r2"] = pd.to_numeric(df["r2"], errors='coerce')

# Посчитать дополнительные параметры
df["roi_dd"] = df.netProfitPercent / df.maxStrategyDrawDownPercent
df['roi_dd'] = df['roi_dd'].round(decimals=4)

df = df[df.timeframe < 100]

# Фильтры
# df = df[df.totalTrades > 100]
# df = df[df.totalTrades < 1000]
df = df[df.maxStrategyDrawDownPercent < 0.50]
# df = df[df.roi_dd > 3]
df = df[df.r2 > 0.5]

df = df[df.instrument == "ZO1!"]

# df = df[df.bars_in_trades < 11000]
# df = df[df.bars_in_trades > 5000]



df2 = df.groupby(["strategy", "imageUrl"]).agg(
    cnt=pd.NamedAgg(column="roi_dd", aggfunc="count"),
    mean_roi_dd=pd.NamedAgg(column="roi_dd", aggfunc="mean"),
    # mean_dd=pd.NamedAgg(column="maxStrategyDrawDownPercent", aggfunc="mean"),
    mean_pf=pd.NamedAgg(column="profitFactor", aggfunc="mean"),
    mean_r2=pd.NamedAgg(column="r2", aggfunc="mean"),
    mean_trades=pd.NamedAgg(column="totalTrades", aggfunc="mean"),
)
df2 = df2[df2.cnt > 1] 
df2 = df2[df2.mean_pf > 1] 
df2 = df2.sort_values("mean_r2", ascending=False)
df2



# df = df.groupby(["instrument"]).agg(
#     cnt=pd.NamedAgg(column="roi_dd", aggfunc="count"),
#     mean_rel_roi=pd.NamedAgg(column="netProfitPercent", aggfunc="mean"),
#     mean_roi_dd=pd.NamedAgg(column="roi_dd", aggfunc="mean"),
#     mean_r2=pd.NamedAgg(column="r2", aggfunc="mean"),
# )
# df = df[df.cnt > 1]
# df = df[df.mean_roi_dd > 1]
# df.sort_values("cnt", ascending=False)



,,cnt,mean_roi_dd,mean_pf,mean_r2,mean_trades
strategy,imageUrl,,,,,
